# Daily Dose of Data Science

This notebook accompanies the code for RAG demo published in the Daily Dose of Data Science.

Read the article here: [A crash course on RAG - Part 1](https://www.dailydoseofds.com/a-crash-course-on-building-rag-systems-part-1-with-implementations/)

## Set up Asyncio

In [1]:
import nest_asyncio

nest_asyncio.apply()

## Set up the Qdrant vector database

In [2]:
import qdrant_client

collection_name="chat_with_docs"

client = qdrant_client.QdrantClient(
    host="localhost",
    port=6333
)

/Users/rajeshthakur/miniconda3/envs/maf_env/lib/python3.11/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.16.3. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


## Read the documents

In [4]:
from llama_index.core import SimpleDirectoryReader

input_dir_path = './docs'

loader = SimpleDirectoryReader(
            input_dir = input_dir_path,
            required_exts=[".pdf"],
            recursive=True
        )
docs = loader.load_data()

In [5]:
type(docs), len(docs)

(list, 1)

## A function to index data

In [6]:
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex, ServiceContext, StorageContext

def create_index(documents):
    vector_store = QdrantVectorStore(client=client, collection_name=collection_name)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
    )
    return index

## Load the embedding model and index data

In [7]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-large-en-v1.5", trust_remote_code=True)
Settings.embed_model = embed_model

index = create_index(docs)

/Users/rajeshthakur/miniconda3/envs/maf_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Define the prompt template

In [8]:
from llama_index.llms.ollama import Ollama

llm=Ollama(model="llama3.2:3b", request_timeout=120.0)

Settings.llm = llm

## Define the prompt template

In [9]:
from llama_index.core import PromptTemplate

qa_prompt_tmpl_str = (
"Context information is below.\n"
"---------------------\n"
"{context_str}\n"
"---------------------\n"
"Given the context information above I want you to think step by step to answer the query in a crisp manner, incase case you don't know the answer say 'I don't know!'.\n"
"Query: {query_str}\n"
"Answer: "
)

qa_prompt_tmpl = PromptTemplate(qa_prompt_tmpl_str)

## Reranking

In [10]:
from llama_index.core.postprocessor import SentenceTransformerRerank

rerank = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-2-v2", 
    top_n=3
)

## Query the document

In [11]:
query_engine = index.as_query_engine(similarity_top_k=10, node_postprocessors=[rerank])

query_engine.update_prompts(
    {"response_synthesizer:text_qa_template": qa_prompt_tmpl}
)

response = query_engine.query("What exactly is DSPy?")

## Print response

In [12]:
from IPython.display import Markdown, display

display(Markdown(str(response)))

I don't know what DSPy refers to specifically from the context provided. However, based on general knowledge, "DSP" typically stands for Digital Signal Processing. If DSPy were a specific application or technique within DSP, it would require more information to determine its exact nature.

Without additional context or information about where this term is used (e.g., academic papers, industry reports, specific field like audio processing, etc.), I cannot accurately define what DSPy specifically means.